<a href="https://www.kaggle.com/code/avikdas567/localized-medical-protocol-agent-gemma-4-e4b-rag?scriptVersionId=310922527" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Environment Setup

In [1]:
!pip install -q -U transformers accelerate bitsandbytes chromadb sentence-transformers gradio > /dev/null 2>&1
print("Libraries installed successfully!")

Libraries installed successfully!


# Imports & Configurations

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import chromadb
from sentence_transformers import SentenceTransformer
import gradio as gr
import warnings

warnings.filterwarnings('ignore')

# Loading Gemma 4

In [3]:
model_id = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1" 

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading Gemma 4 Model across Dual T4s...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
print("Model loaded successfully!")

Loading Tokenizer...
Loading Gemma 4 Model across Dual T4s...


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Model loaded successfully!


# Initializing the Offline Vector Database

In [4]:
print("Initializing Local Vector Database...")

# Synthetic Edge Medical Database
documents = [
    "Protocol 101A: For severe allergic reactions (anaphylaxis), administer Epinephrine 0.3 mg IM immediately. Monitor airway.",
    "Protocol 204B: In cases of suspected stroke, perform the FAST assessment. Do not administer aspirin until CT scan rules out hemorrhage.",
    "Protocol 305C: For acute asthma exacerbation, provide Albuterol 2.5 mg via nebulizer. Repeat every 20 minutes for up to 3 doses.",
    "Protocol 401D: Post-partum hemorrhage management includes administering Oxytocin 10 units IM and initiating fundal massage.",
    "Protocol 505E: For suspected opioid overdose, administer Naloxone 0.4 mg to 2 mg IV/IM/SubQ immediately."
]

local_embedder_path = "/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2" 
embedder = SentenceTransformer(local_embedder_path)

embeddings = embedder.encode(documents)

# Initialize ChromaDB
client = chromadb.Client()
collection = client.create_collection(name="medical_protocols")

# Populate the database
for i, doc in enumerate(documents):
    collection.add(
        embeddings=[embeddings[i].tolist()],
        documents=[doc],
        ids=[f"protocol_{i}"]
    )
print("Database ready. Zero internet required!")

Initializing Local Vector Database...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Database ready. Zero internet required!


# The Agentic RAG Pipeline

In [5]:
import re

def medical_agent(query):
    # Agentic Retrieval: Find the most relevant protocol offline
    query_emb = embedder.encode(query).tolist()
    results = collection.query(query_embeddings=[query_emb], n_results=1)
    
    if not results['documents'][0]:
        return "No relevant medical protocol found in the local database."
        
    context = results['documents'][0][0]

    # Prompting
    prompt = f"""<start_of_turn>user
You are an AI Medical Assistant. Use the protocol below to answer the query.
Protocol: {context}
Query: {query}
Instructions: Provide ONLY the protocol instructions. Do not explain your thinking.<end_of_turn>
<start_of_turn>model
The immediate steps are: """

    # Generation using our offline Gemma 4 E4B model
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    outputs = model.generate(
        **inputs, 
        max_new_tokens=100, 
        temperature=0.1, 
        do_sample=True,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id
    )
    
    # Decoding
    raw_response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Robust Extraction
    if "The immediate steps are:" in raw_response:
        final_answer = raw_response.split("The immediate steps are:")[-1].strip()
    else:
        final_answer = raw_response.split("model")[-1].strip()
    
    final_answer = final_answer.split("thought")[0].split("The user wants")[0].strip()

    return final_answer if final_answer else "Error: Model produced empty output."

# Test the pipeline
print("--- TEST OUTPUT START ---")
print(medical_agent("Patient comes in with severe allergic reaction."))
print("--- TEST OUTPUT END ---")

--- TEST OUTPUT START ---
1. Administer Epinephrine 0.3 mg IM immediately. 2. Monitor airway.
--- TEST OUTPUT END ---


# Launching the Live Demo UI

In [6]:
import gradio as gr
import time

# Define the UI Interface
demo = gr.Interface(
    fn=medical_agent,
    inputs=gr.Textbox(
        lines=3, 
        placeholder="Enter patient symptoms or situation (e.g., 'Patient showing signs of stroke')..."
    ),
    outputs=gr.Textbox(label="Gemma 4 Edge Protocol Recommendation"),
    title="🏥 MediEdge: Edge-Native Protocol Agent",
    description="**Powered by Gemma 4.** A localized, privacy-first Agentic RAG system designed for edge medical sites. It grounds its answers strictly in local medical protocols.",
    theme="soft"
)

# Launch with prevent_thread_lock=True
# This allows the code to continue to the next line after starting the server
demo.launch(share=True, prevent_thread_lock=True)

print("Server is starting... capturing public URL for logs. Please wait 30 seconds.")
time.sleep(30)

demo.close()
print("Notebook execution complete. Save version successful.")

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://79f39789b86eaccc58.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Server is starting... capturing public URL for logs. Please wait 30 seconds.
Closing server running on port: 7860
Notebook execution complete. Save version successful.
